# Cartographie T2 en IRM : moindres carrés vs réseau de neurones (PyTorch)

En IRM quantitative, on ne veut pas seulement une jolie image, on veut mesurer une propriété physique de chaque tissu. Ici, c'est le temps de relaxation T2 (en millisecondes), pixel par pixel.

Ce que je fais dans ce notebook :
1. Comprendre le modèle physique du signal (une simple exponentielle décroissante).
2. Estimer T2 avec une méthode classique : **les moindres carrés** (même outil que la pseudo-inverse).
3. Estimer T2 avec un **petit réseau de neurones (MLP) en PyTorch**, entraîné sur des données **synthétiques**.
4. Comparer les deux sur un fantôme de cerveau simplifié.
5. Voir le **piège** : un modèle entraîné sur du synthétique peut se tromper dès qu'il sort de ce qu'il a vu.

> Toutes les données sont simulées. Les valeurs de T2 sont des ordres de grandeur simplifiés, pas des valeurs cliniques.

In [ ]:
import numpy as np                      # calcul numerique
import matplotlib.pyplot as plt         # figures
import torch                            # PyTorch
import torch.nn as nn                   # briques de reseaux de neurones
import os

np.random.seed(0)                       # reproductibilite numpy
torch.manual_seed(0)                    # reproductibilite PyTorch
os.makedirs("figures", exist_ok=True)   # dossier pour sauvegarder les figures

## 1. Le modèle physique : décroissance T2

Dans une séquence **multi-échos de spin**, on mesure le signal d'un même pixel à plusieurs **temps d'écho** $TE_1, \dots, TE_8$. Le signal décroît de façon exponentielle :

$$S(TE) = S_0 \, e^{-TE / T_2}$$

- $S_0$ : signal de départ (dépend de la densité de protons du tissu)
- $T_2$ : vitesse de décroissance. **C'est ce qu'on veut mesurer.**

Chaque tissu a son propre T2 : c'est ce qui donne le contraste en IRM.

In [ ]:
TE = np.arange(10, 90, 10).astype(float)    # 8 temps d'echo : 10, 20, ..., 80 ms
print("Temps d'echo (ms) :", TE)

def signal_T2(S0, T2, TE):
    # Modele physique : S(TE) = S0 * exp(-TE / T2)
    return S0 * np.exp(-TE / T2)

# Quelques tissus (valeurs simplifiees, ordre de grandeur)
tissus = {"Substance blanche (T2=70)": 70,
          "Substance grise (T2=100)": 100,
          "Lesion (T2=150)": 150}

TE_fin = np.linspace(0, 100, 200)          # grille fine pour tracer des courbes lisses
plt.figure(figsize=(6, 4))
for nom, T2 in tissus.items():
    plt.plot(TE_fin, signal_T2(1.0, T2, TE_fin), label=nom)
    plt.plot(TE, signal_T2(1.0, T2, TE), "o", color=plt.gca().lines[-1].get_color())
plt.xlabel("TE (ms)"); plt.ylabel("Signal (S0 = 1)")
plt.title("Decroissance T2 : les points = les 8 echos mesures")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig("figures/1_decroissance_T2.png", dpi=150); plt.show()

## 2. Le bruit

Une vraie mesure est bruitée. En IRM, le signal brut est un nombre **complexe** (partie réelle + imaginaire), chacune avec un bruit gaussien de même écart-type $\sigma$. L'image qu'on regarde est le **module** de ce nombre :

$$S_{mesure} = \sqrt{(S + n_1)^2 + n_2^2}, \qquad n_1, n_2 \sim \mathcal{N}(0, \sigma^2)$$

Une ligne de numpy suffit à le simuler. Conséquence pratique : le signal mesuré est **toujours positif**, et à faible signal il est légèrement **surestimé**.

In [ ]:
def ajouter_bruit(S, sigma):
    # Module d'un signal complexe bruite (bruit gaussien sur partie reelle et imaginaire)
    n1 = np.random.normal(0, sigma, S.shape)
    n2 = np.random.normal(0, sigma, S.shape)
    return np.sqrt((S + n1) ** 2 + n2 ** 2)

S_propre = signal_T2(1.0, 100, TE)
S_bruite = ajouter_bruit(S_propre, sigma=0.03)
plt.figure(figsize=(6, 4))
plt.plot(TE, S_propre, "o-", label="signal propre")
plt.plot(TE, S_bruite, "x--", label="signal bruite (sigma = 0.03)")
plt.xlabel("TE (ms)"); plt.ylabel("Signal"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Un pixel de substance grise (T2 = 100 ms)"); plt.tight_layout(); plt.show()

## 3. Méthode classique : moindres carrés sur le logarithme

On prend le logarithme du modèle :

$$\ln S(TE) = \ln S_0 - \frac{1}{T_2}\, TE$$

C'est une droite en fonction de $TE$. Avec les inconnues $\theta = (\ln S_0,\; 1/T_2)$ on écrit pour les 8 échos :

$$\underbrace{\begin{pmatrix} 1 & -TE_1 \\ \vdots & \vdots \\ 1 & -TE_8 \end{pmatrix}}_{A} \theta = \begin{pmatrix} \ln S_1 \\ \vdots \\ \ln S_8 \end{pmatrix}$$

8 équations, 2 inconnues : on résout au sens des **moindres carrés**, $\theta = A^{+} \ln S$ avec la **pseudo-inverse** de Moore-Penrose.

Astuce : la matrice $A$ est **la même pour tous les pixels**. On calcule $A^{+}$ une seule fois, puis un produit matriciel traite tous les pixels d'un coup.

In [ ]:
A = np.column_stack([np.ones_like(TE), -TE])   # matrice 8 x 2
A_pinv = np.linalg.pinv(A)                      # pseudo-inverse 2 x 8, calculee une seule fois

def T2_moindres_carres(S):
    # S : tableau (nb_pixels, 8). Renvoie T2 estime pour chaque pixel.
    logS = np.log(np.clip(S, 1e-6, None))       # clip : eviter log(0)
    theta = logS @ A_pinv.T                     # (nb_pixels, 2) : [ln S0, 1/T2]
    inv_T2 = np.clip(theta[:, 1], 1e-4, None)   # 1/T2 doit etre positif
    return 1.0 / inv_T2

print("T2 vrai = 100 ms | estime (signal propre)  :", T2_moindres_carres(S_propre[None, :])[0])
print("T2 vrai = 100 ms | estime (signal bruite) :", T2_moindres_carres(S_bruite[None, :])[0])

## 4. Données d'entraînement synthétiques

Pour entraîner un réseau, il faut beaucoup d'exemples **(signal → T2 vrai)**. On les fabrique avec notre modèle physique :
- T2 tiré au hasard entre **20 et 300 ms**,
- $S_0$ entre 0.5 et 1.5,
- bruit $\sigma$ entre **0.01 et 0.05**.

**Normalisation** (important pour un réseau) :
- en entrée, on divise chaque signal par son premier écho, pour que le réseau ne dépende pas de $S_0$ ;
- en sortie, on prédit $T_2 / 300$, un nombre entre 0 et 1.

In [ ]:
T2_MAX = 300.0

def generer_donnees(n, T2_min=20, T2_max=300, sigma_min=0.01, sigma_max=0.05):
    T2 = np.random.uniform(T2_min, T2_max, n)
    S0 = np.random.uniform(0.5, 1.5, n)
    sigma = np.random.uniform(sigma_min, sigma_max, n)
    S = S0[:, None] * np.exp(-TE[None, :] / T2[:, None])   # (n, 8) signaux propres
    S = ajouter_bruit(S, sigma[:, None])                    # (n, 8) signaux bruites
    return S, T2, sigma

def normaliser(S):
    return S / S[:, :1]                                     # divise chaque ligne par son 1er echo

S_train, T2_train, _ = generer_donnees(50_000)
S_test,  T2_test,  sigma_test = generer_donnees(5_000)

# Conversion en tensors PyTorch (float32)
X_train = torch.tensor(normaliser(S_train), dtype=torch.float32)
y_train = torch.tensor(T2_train / T2_MAX, dtype=torch.float32).unsqueeze(1)   # forme (n, 1)
X_test  = torch.tensor(normaliser(S_test), dtype=torch.float32)
print(X_train.shape, y_train.shape)

## 5. Le réseau de neurones (MLP) en PyTorch

Par rapport à mon projet d'introduction (polynôme et `autograd` à la main), trois nouveautés :
- **`nn.Sequential`** : on empile des couches au lieu d'écrire les poids à la main ;
- **`torch.optim.Adam`** : l'optimiseur fait la mise à jour des poids à notre place (plus besoin de `with torch.no_grad()` ni de `grad.zero_()` manuels) ;
- **mini-batchs** : à chaque pas, on n'utilise que 256 exemples tirés au hasard, ce qui est plus rapide.

Architecture : 8 entrées (les échos) → 64 neurones → 64 neurones → 1 sortie (T2/300), avec la fonction d'activation ReLU entre les couches.

In [ ]:
modele = nn.Sequential(
    nn.Linear(8, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 1),
)
optimiseur = torch.optim.Adam(modele.parameters(), lr=1e-3)
perte_fn = nn.MSELoss()                         # erreur quadratique moyenne

n_epochs, batch = 30, 256
historique = []
for epoch in range(n_epochs):
    permutation = torch.randperm(len(X_train))  # on melange les exemples a chaque epoch
    for i in range(0, len(X_train), batch):
        idx = permutation[i:i + batch]
        y_pred = modele(X_train[idx])           # passe avant
        perte = perte_fn(y_pred, y_train[idx])  # calcul de la perte
        optimiseur.zero_grad()                  # remise a zero des gradients
        perte.backward()                        # autograd : calcul des gradients
        optimiseur.step()                       # mise a jour des poids
    historique.append(perte.item())
    if epoch % 5 == 0:
        print(f"epoch {epoch:2d}  perte = {perte.item():.5f}")

plt.figure(figsize=(6, 3.5))
plt.semilogy(historique); plt.xlabel("epoch"); plt.ylabel("perte (MSE)")
plt.title("Courbe d'apprentissage"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 6. Comparaison sur le jeu de test

Même jeu de test (5 000 signaux jamais vus) pour les deux méthodes. On mesure l'**erreur relative** $|T_2^{estimé} - T_2^{vrai}| / T_2^{vrai}$.

In [ ]:
def T2_reseau(S):
    X = torch.tensor(normaliser(S), dtype=torch.float32)
    with torch.no_grad():                        # pas besoin de gradients pour predire
        return modele(X).squeeze(1).numpy() * T2_MAX

T2_mc  = T2_moindres_carres(S_test)
T2_mlp = T2_reseau(S_test)

def erreur_relative(estime, vrai):
    return np.abs(estime - vrai) / vrai

print(f"Erreur relative mediane - moindres carres : {np.median(erreur_relative(T2_mc, T2_test)):.1%}")
print(f"Erreur relative mediane - reseau (MLP)    : {np.median(erreur_relative(T2_mlp, T2_test)):.1%}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, est, titre in zip(axes, (T2_mc, T2_mlp), ("Moindres carres", "Reseau (MLP)")):
    ax.scatter(T2_test, est, s=2, alpha=0.3)
    ax.plot([0, 300], [0, 300], "r--", label="estimation parfaite")
    ax.set_xlim(0, 320); ax.set_ylim(0, 450)
    ax.set_xlabel("T2 vrai (ms)"); ax.set_title(titre); ax.grid(alpha=0.3)
axes[0].set_ylabel("T2 estime (ms)"); axes[0].legend()
plt.tight_layout(); plt.savefig("figures/2_comparaison_test.png", dpi=150); plt.show()

**À observer :**
- Les moindres carrés se dispersent de plus en plus quand T2 augmente. Avec des échos qui s'arrêtent à 80 ms, un T2 long décroît très peu sur la fenêtre de mesure, donc le bruit pèse lourd.
- Le réseau est moins dispersé, mais il **sous-estime** les T2 proches de 300 ms : au bord de sa plage d'apprentissage, il « n'ose » pas prédire des valeurs extrêmes et se rabat vers le milieu. C'est un premier signe de sa limite.

## 7. Application à un fantôme de cerveau

On construit une image 128×128 avec des ellipses : substance grise, substance blanche, deux ventricules (liquide cérébrospinal, LCR) et une petite lésion. Chaque région a son T2. On simule les 8 images d'écho, puis on estime la carte T2 avec les deux méthodes.

Attention : le **LCR a un T2 très long** (de l'ordre de 1 000 ms et plus), **bien au-delà** des 300 ms vus pendant l'entraînement.

In [ ]:
def ellipse(N, cy, cx, ry, rx):
    yy, xx = np.mgrid[:N, :N]
    return ((yy - cy) / ry) ** 2 + ((xx - cx) / rx) ** 2 <= 1

N = 128
T2_carte = np.zeros((N, N)); S0_carte = np.zeros((N, N))
regions = [  # (masque, T2 en ms, S0)       -- valeurs simplifiees
    (ellipse(N, 64, 64, 50, 40), 100, 0.8),   # substance grise
    (ellipse(N, 64, 64, 38, 29), 70, 0.7),    # substance blanche
    (ellipse(N, 60, 52, 14, 5), 1000, 1.0),   # ventricule gauche (LCR)
    (ellipse(N, 60, 76, 14, 5), 1000, 1.0),   # ventricule droit (LCR)
    (ellipse(N, 88, 72, 6, 6), 150, 0.85),    # lesion
]
for masque, T2, S0 in regions:
    T2_carte[masque] = T2; S0_carte[masque] = S0
cerveau = T2_carte > 0

# 8 images d'echo bruitees : tableau (N, N, 8)
images = S0_carte[..., None] * np.exp(-TE / np.where(cerveau, T2_carte, 1.0)[..., None])
images = ajouter_bruit(images, sigma=0.02)

# Estimation pixel par pixel (uniquement dans le cerveau)
pixels = images[cerveau]                       # (nb_pixels, 8)
carte_mc = np.zeros((N, N));  carte_mc[cerveau]  = T2_moindres_carres(pixels)
carte_mlp = np.zeros((N, N)); carte_mlp[cerveau] = T2_reseau(pixels)

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, img, titre in zip(axes, (images[..., 0], T2_carte, carte_mc, carte_mlp),
                          ("Image echo 1 (TE=10 ms)", "T2 vrai", "T2 moindres carres", "T2 reseau (MLP)")):
    vmax = None if titre.startswith("Image") else 300
    im = ax.imshow(img, cmap="gray" if vmax is None else "viridis", vmin=0, vmax=vmax)
    ax.set_title(titre); ax.axis("off"); plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.savefig("figures/3_fantome_cartes_T2.png", dpi=150); plt.show()

# Valeur mediane estimee dans chaque tissu (chaque tissu a un T2 different)
for nom, T2 in [("Subst. blanche", 70), ("Subst. grise", 100), ("Lesion", 150), ("LCR", 1000)]:
    zone = T2_carte == T2
    print(f"{nom:15s} T2 vrai = {T2:5d} ms | MC : {np.median(carte_mc[zone]):7.1f} | MLP : {np.median(carte_mlp[zone]):6.1f}")

**À observer :**
- Substance blanche, substance grise et lésion : les deux méthodes retrouvent des valeurs proches des vraies.
- **Ventricules (LCR, T2 = 1 000 ms)** : l'échelle de couleur est coupée à 300 ms, donc il faut **lire les valeurs imprimées**. Les moindres carrés retrouvent un ordre de grandeur correct, parce qu'ils suivent l'équation physique. Le réseau reste **bloqué sous 300 ms** : il n'a jamais vu de T2 plus long, et il répond quand même, sans prévenir.

## 8. Le piège : du synthétique au réel

Un réseau entraîné sur des données synthétiques ne connaît que **ce qu'on lui a montré**. On le teste ici hors de sa zone d'entraînement de deux façons :
1. **bruit plus fort** qu'à l'entraînement ($\sigma$ = 0.10 au lieu de 0.01–0.05), comme un scanner moins bon ou une acquisition accélérée ;
2. **T2 hors plage** (300–600 ms au lieu de 20–300 ms), comme un tissu qu'on n'avait pas prévu.

In [ ]:
scenarios = {
    "Comme a l'entrainement": dict(),
    "Bruit plus fort (sigma=0.10)": dict(sigma_min=0.10, sigma_max=0.10),
    "T2 hors plage (300-600 ms)": dict(T2_min=300, T2_max=600),
}
resultats = {}
for nom, params in scenarios.items():
    S, T2_vrai, _ = generer_donnees(5000, **params)
    resultats[nom] = (np.median(erreur_relative(T2_moindres_carres(S), T2_vrai)),
                      np.median(erreur_relative(T2_reseau(S), T2_vrai)))
    print(f"{nom:30s} | MC : {resultats[nom][0]:6.1%} | MLP : {resultats[nom][1]:6.1%}")

noms = list(resultats); x = np.arange(len(noms))
plt.figure(figsize=(8, 4))
plt.bar(x - 0.2, [resultats[n][0] for n in noms], 0.4, label="Moindres carres")
plt.bar(x + 0.2, [resultats[n][1] for n in noms], 0.4, label="Reseau (MLP)")
plt.xticks(x, noms); plt.ylabel("Erreur relative mediane")
plt.title("Que se passe-t-il hors de la zone d'entrainement ?")
plt.legend(); plt.grid(axis="y", alpha=0.3); plt.tight_layout()
plt.savefig("figures/4_hors_distribution.png", dpi=150); plt.show()

**Ce que je retiens :**
- Dans sa zone d'entraînement, le réseau est **précis et rapide**.
- **Toutes les sorties de zone ne se valent pas.** Un bruit un peu plus fort dégrade les deux méthodes, mais le réseau tient encore. En revanche, des **T2 hors de la plage apprise** le font échouer nettement : il ne peut pas prédire une valeur qu'il n'a jamais vue.
- Le réseau donne **toujours** une réponse, même quand elle n'a pas de sens physique. Rien ne signale l'erreur.
- La méthode classique est moins précise, mais elle repose directement sur l'équation physique et ne dépend pas d'une plage apprise.
- C'est l'un des grands risques pour passer d'un modèle entraîné sur du **synthétique** à des **données cliniques réelles** : le vrai scanner produit du bruit, des artefacts et des tissus que la simulation n'a pas prévus. D'où l'importance de valider sur des données réelles.

## 9. Pour aller plus loin (à faire moi-même)
- [ ] Élargir la plage d'entraînement (T2 jusqu'à 2 000 ms, bruit jusqu'à 0.15) : est-ce que tout se règle ?
- [ ] Donner au réseau une **estimation du bruit** en entrée supplémentaire.
- [ ] Remplacer la régression log-linéaire par un **ajustement non linéaire** (`scipy.optimize.curve_fit`) et comparer.
- [ ] Faire varier la taille du réseau (16, 64, 256 neurones) et regarder l'effet.
- [ ] Passer au **T1** avec une séquence d'inversion-récupération : $S = S_0 \,|1 - 2e^{-TI/T_1}|$.